In [3]:
!pip install sentencepiece datasets -q

In [4]:
import os
from datasets import load_dataset
import sentencepiece as spm

print('SentencePiece version:', spm.__version__)

SentencePiece version: 0.2.1


## Prepare Training Text
Use a small slice of WikiText-2 to keep training quick.

In [5]:
dataset = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train[:5%]')
print('Samples:', len(dataset))

train_file = 'spm_train.txt'
with open(train_file, 'w', encoding='utf-8') as f:
    for row in dataset['text']:
        if row.strip():
            f.write(row.strip() + '\n')
print('Wrote training text to', train_file)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Samples: 1836
Wrote training text to spm_train.txt


## Train SentencePiece (BPE)

In [13]:
model_prefix = 'toy_bpe'
vocab_size = 800

spm.SentencePieceTrainer.Train(
    input=train_file,
    model_prefix=model_prefix,
    vocab_size=vocab_size,
    model_type='bpe',
    character_coverage=1.0,
    pad_id=0, unk_id=1, bos_id=2, eos_id=3
)

print('Trained files:', model_prefix + '.model', model_prefix + '.vocab')
print('Vocab size (requested):', vocab_size)

Trained files: toy_bpe.model toy_bpe.vocab
Vocab size (requested): 800


## Encode / Decode Examples

In [14]:
sp = spm.SentencePieceProcessor(model_file=model_prefix + '.model')

samples = [
    'Transformers changed NLP.',
    'SentencePiece handles languages without whitespace.',
    'Tokenizers control vocabulary size and OOV handling.'
]

for text in samples:
    ids = sp.encode(text, out_type=int)
    pieces = sp.encode(text, out_type=str)
    decoded = sp.decode(ids)
    print('\nOriginal:', text)
    print('Pieces:', pieces)
    print('IDs   :', ids)
    print('Decoded:', decoded)


Original: Transformers changed NLP.
Pieces: ['▁T', 'ran', 's', 'form', 'ers', '▁ch', 'an', 'g', 'ed', '▁N', 'L', 'P', '.']
IDs   : [44, 255, 643, 488, 136, 192, 30, 651, 14, 114, 685, 676, 657]
Decoded: Transformers changed NLP.

Original: SentencePiece handles languages without whitespace.
Pieces: ['▁S', 'ent', 'ence', 'P', 'ie', 'ce', '▁h', 'and', 'les', '▁l', 'an', 'g', 'u', 'ag', 'es', '▁with', 'out', '▁wh', 'it', 'es', 'p', 'ace', '.']
IDs   : [54, 51, 359, 676, 253, 108, 42, 141, 345, 63, 30, 651, 648, 140, 27, 91, 320, 118, 21, 27, 652, 392, 657]
Decoded: SentencePiece handles languages without whitespace.

Original: Tokenizers control vocabulary size and OOV handling.
Pieces: ['▁T', 'ok', 'en', 'iz', 'ers', '▁cont', 'ro', 'l', '▁v', 'oc', 'ab', 'ul', 'ary', '▁s', 'iz', 'e', '▁and', '▁O', 'O', 'V', '▁h', 'and', 'l', 'ing', '.']
IDs   : [44, 315, 24, 306, 136, 294, 40, 646, 185, 156, 188, 100, 237, 13, 306, 636, 35, 180, 697, 702, 42, 141, 646, 33, 657]
Decoded: Tokenizers contr

## Save + Reload Check

In [15]:
# Re-load processor and ensure deterministic behavior
sp2 = spm.SentencePieceProcessor(model_file=model_prefix + '.model')
ids = sp2.encode(samples[0], out_type=int)
print('Reloaded encoding matches original?', ids == sp.encode(samples[0], out_type=int))

Reloaded encoding matches original? True


## Takeaways
- BPE merges frequent character pairs to balance vocab size and coverage.
- SentencePiece trains from raw text (no pre-tokenization needed).
- Check for OOV handling via `unk_id`; adjust vocab_size for your domain.